# Overview

1. We start with the initial TS structure.
2. We run metadynamics from that structure, using PLUMED to orchestrate `orbmol-v2`. What's metadynamics? It's a variation on running trajectories where you bias certain outcomes to avoid getting stuck in a local minima.
3. We run a custom sampling algorithm to select the most salient structures from the metadynamics run (essentially eucledian distance from already selected structures to make sure they're sufficiently different).
4. Run DFT calculations on these salient structures to get the energies and forces.
5. We split these DFT calculations into a training and validation dataset. We'll call these combined to be dataset A. We'll also do this a second time to make dataset B. That way both A and B contain all DFT calculations, but the partitioning is (randomly) different.
6. Take the current orbmol model and fine-tune it twice, once with A and once with B. This is done independently, as in we'll have an orbmol A and an orbmol B, each forking from the current orbmol model.
7. Run metadynamics with orbmol A.
8. Calculate energies and forces with orbmol B on every structure from orbmol A's metadynamics. This gives us the difference between orbmol A's predictions and orbmol B's predictions.
9. Search through and find all structures that significantly deviate between orbmol A's predicted energies and orbmol B's predicted energies.
10. Run the same sampling algorithm as #3 on these deviant structures to further narrow relevant deviants.
11. Loop back to #4, with the deviant structures as the new salient structures.


In [2]:
import os
import torch
from ase import Atoms, units

if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Using NVIDIA GPU: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    num_cores = len(os.sched_getaffinity(0))
    torch.set_num_threads(num_cores)
    os.environ["OMP_NUM_THREADS"] = str(num_cores)
    os.environ["MKL_NUM_THREADS"] = str(num_cores)
    print(f"Using CPU with {num_cores} threads")

Using CPU with 16 threads


In [3]:
# Load model.

from orb_models.forcefield import pretrained
from orb_models.forcefield.inference.calculator import ORBCalculator

orbff, atoms_adapter = pretrained.orbmol_v2(
    device=device,
    precision="float32-high",
)
model = ORBCalculator(orbff, atoms_adapter=atoms_adapter, device=device)

/home/mason/research/gold-research/mlip/.venv/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
Warp CUDA warning: Could not find or load the NVIDIA CUDA driver. GPU execution will not be available.


Warp 1.18.0 initialized:
   CUDA Toolkit 13.4, CUDA driver not available (NVRTC compilation available)
   Devices:
     "cpu"      : "CPU"
   Kernel cache:
     /home/mason/.cache/warp/1.18.0


/home/mason/research/gold-research/mlip/.venv/lib/python3.12/site-packages/orb_models/common/training/util.py:20: UserWarning: Setting global torch default dtype to torch.float32.
  warnings.warn(f"Setting global torch default dtype to {torch_dtype}.")


In [4]:
# Load and configure the XYZ structure.

import ase.io

atoms = ase.io.read("start.xyz")
if isinstance(atoms, list):
    atoms = atoms[-1]
atoms.info.update({"spin": 1, "charge": 1})
atoms.set_pbc(False) # No periodic boundary.

In [ ]:
from ase.md import MDLogger
from ase.md.bussi import Bussi
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution

temperature_K = 298.15
timestep = 0.5  # fs
taut = 100 # fs
atoms.calc = model
MaxwellBoltzmannDistribution(atoms, temperature_K=298.15)
dyn = Bussi(atoms, timestep * units.fs, temperature_K, taut * units.fs)

# Save the trajectory
traj = ase.io.Trajectory('mace_mtd.traj', 'w', atoms)
dyn.attach(traj.write, interval=1)
# Set up a logger to monitor the simulation
dyn.attach(MDLogger(dyn, atoms, 'md_equil.log', header=True, stress=False), interval=1)

/tmp/ipykernel_578601/2793565621.py:9: DeprecationWarning: Use thermalize_momenta
  MaxwellBoltzmannDistribution(atoms, temperature_K=298.15)


In [6]:
# Run dynamics
nsteps = 50
dyn.run(nsteps)

# Cleanup
traj.close()

Module nvalchemiops.neighbors.neighbor_utils 32c40ea load on device 'cpu' took 21.39 ms  (cached)
Module nvalchemiops.neighbors.naive 51a3e5a load on device 'cpu' took 1.84 ms  (cached)


In [7]:
with ase.io.Trajectory('mace_mtd.traj', 'r') as traj_read:
    ase.io.write('mace_mtd.xyz', traj_read, plain=True)